# B.3 — Cohen's Kappa: Judge vs Human

Calibrate LLM judge against human labels on 10 pairs.

In [1]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

human = pd.read_csv('human_labels.csv')['human_winner'].tolist()
judge_df = pd.read_csv('pairwise_results.csv').sample(10, random_state=42).reset_index(drop=True)
judge = judge_df['winner_after_swap'].tolist()

def norm(x):
    return str(x).strip().lower()

human = [norm(h) for h in human]
judge = [norm(j) for j in judge]

print(f'Human labels: {human}')
print(f'Judge labels: {judge}')
kappa = cohen_kappa_score(human, judge)
print(f"\nCohen's kappa: {kappa:.3f}")


Human labels: ['b', 'b', 'a', 'b', 'tie', 'a', 'tie', 'b', 'b', 'tie']
Judge labels: ['a', 'a', 'a', 'a', 'tie', 'a', 'tie', 'b', 'a', 'a']

Cohen's kappa: 0.333


## Interpretation

| Kappa | Agreement | Action |
|---|---|---|
| < 0 | Worse than chance | Judge is broken |
| 0.0 - 0.2 | Slight | Cannot be trusted |
| 0.2 - 0.4 | Fair | Still weak |
| 0.4 - 0.6 | Moderate | OK for monitoring |
| 0.6 - 0.8 | Substantial | Production-ready |
| > 0.8 | Almost perfect | Rare |

In [2]:
if kappa < 0.2:
    print('WORSE than chance - judge is broken')
elif kappa < 0.4:
    print('Slight/Fair agreement - cannot be trusted, re-check prompt + re-label')
elif kappa < 0.6:
    print('Moderate - OK monitoring, not production yet')
elif kappa < 0.8:
    print('Substantial - production-ready')
else:
    print('Almost perfect')

if kappa < 0.6:
    print('\n## Root cause analysis (kappa < 0.6)')
    print('- Position bias suspect: re-check column run1_winner balance')
    print('- Length bias suspect: correlate len_diff vs winner_after_swap')
    print('- Style bias: judge may prefer formal vs casual Vietnamese')
    print('- Sample size (n=10) too small for reliable estimate; aim for 30+')


Slight/Fair agreement - cannot be trusted, re-check prompt + re-label

## Root cause analysis (kappa < 0.6)
- Position bias suspect: re-check column run1_winner balance
- Length bias suspect: correlate len_diff vs winner_after_swap
- Style bias: judge may prefer formal vs casual Vietnamese
- Sample size (n=10) too small for reliable estimate; aim for 30+
